# 04 — Exploratory Data Analysis & Train/Test Split
**Task:** Sentiment Analysis  
**Covers:** Label distribution, text length, vocabulary, word clouds, TF-IDF top terms, stratified splitting

---

In [ ]:
!pip install -q datasets pandas matplotlib seaborn wordcloud scikit-learn nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.download('stopwords', quiet=True)

%matplotlib inline
sns.set_theme(style='whitegrid')

# Load SST-2 dataset
from datasets import load_dataset
ds = load_dataset("glue", "sst2", split="train")
df = pd.DataFrame({'text': ds['sentence'], 'label': ds['label']})
df['sentiment'] = df['label'].map({0: 'Negative', 1: 'Positive'})
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#E53935', '#43A047'], edgecolor='white')
for i, (v, c) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, c + 50, str(c), ha='center', fontweight='bold')
axes[0].set_title('Label Counts', fontsize=13)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            autopct='%1.1f%%', colors=['#E53935', '#43A047'],
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Label Proportions', fontsize=13)

plt.suptitle('Class Distribution — SST-2', fontsize=14)
plt.tight_layout()
plt.show()

print(counts.to_frame('count').assign(pct=lambda x: (x['count']/x['count'].sum()*100).round(2)))

## 2. Text Length Analysis

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_len'] = df['text'].str.split().str.len()

print("=== Character Length ===")
print(df.groupby('sentiment')['char_len'].describe().round(1))
print("\n=== Word Length ===")
print(df.groupby('sentiment')['word_len'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, ax, title in zip(['char_len', 'word_len'], axes, ['Character Length', 'Word Count']):
    for sentiment, color in zip(['Positive', 'Negative'], ['#43A047', '#E53935']):
        subset = df[df['sentiment'] == sentiment][col]
        ax.hist(subset, bins=40, alpha=0.6, label=sentiment, color=color, edgecolor='none')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(title)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Text Length Distribution by Sentiment', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Most Frequent Words

In [ ]:
from nltk.corpus import stopwords
STOP = set(stopwords.words('english'))

def top_words(texts, n=20):
    words = ' '.join(texts).lower().split()
    words = [w for w in words if w.isalpha() and w not in STOP]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, sentiment, color in zip(axes, ['Positive', 'Negative'], ['#43A047', '#E53935']):
    subset = df[df['sentiment'] == sentiment]['text']
    words, freqs = zip(*top_words(subset, 20))
    ax.barh(words[::-1], freqs[::-1], color=color, alpha=0.8)
    ax.set_title(f'Top 20 Words — {sentiment}', fontsize=13)
    ax.set_xlabel('Frequency')

plt.suptitle('Most Frequent Words by Sentiment', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, sentiment, colormap in zip(axes, ['Positive', 'Negative'], ['Greens', 'Reds']):
    text = ' '.join(df[df['sentiment'] == sentiment]['text'])
    wc = WordCloud(width=700, height=350, background_color='white',
                   stopwords=STOP, colormap=colormap,
                   max_words=100, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sentiment} Reviews', fontsize=14, fontweight='bold')

plt.suptitle('Word Clouds by Sentiment', fontsize=15)
plt.tight_layout()
plt.show()

## 5. TF-IDF — Distinctive Terms per Class

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df['text'])

feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, lbl, sentiment, color in zip(axes, [1, 0], ['Positive', 'Negative'], ['#43A047', '#E53935']):
    mask = df['label'].values == lbl
    mean_tfidf = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-20:][::-1]
    ax.barh(feature_names[top_idx][::-1], mean_tfidf[top_idx][::-1], color=color, alpha=0.85)
    ax.set_title(f'Top TF-IDF Terms — {sentiment}', fontsize=12)
    ax.set_xlabel('Mean TF-IDF Score')

plt.tight_layout()
plt.show()

## 6. Train / Validation / Test Split

In [ ]:
# Stratified 70 / 15 / 15 split
X = df['text'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train  : {len(X_train):>6} samples  | {np.mean(y_train==1)*100:.1f}% Positive")
print(f"Val    : {len(X_val):>6} samples  | {np.mean(y_val==1)*100:.1f}% Positive")
print(f"Test   : {len(X_test):>6} samples  | {np.mean(y_test==1)*100:.1f}% Positive")

# Verify stratification
assert abs(np.mean(y_train==1) - np.mean(y==1)) < 0.02, "Stratification failed!"
print("\n✓ Class proportions preserved across splits.")

In [ ]:
# Visualise split proportions
splits = {'Train': y_train, 'Val': y_val, 'Test': y_test}
fig, ax = plt.subplots(figsize=(8, 4))

bottom = np.zeros(3)
for lbl, label_name, color in zip([1, 0], ['Positive', 'Negative'], ['#43A047', '#E53935']):
    counts = [np.sum(y == lbl) for y in splits.values()]
    ax.bar(splits.keys(), counts, bottom=bottom, label=label_name, color=color, alpha=0.85)
    bottom += counts

ax.set_title('Split Composition', fontsize=13)
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Save Splits

In [ ]:
for name, X_, y_ in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    pd.DataFrame({'text': X_, 'label': y_}).to_csv(f'/tmp/sentiment_{name}.csv', index=False)
    print(f"Saved /tmp/sentiment_{name}.csv  ({len(X_)} rows)")

---
**Next Notebook →** `05_machine_learning.ipynb`